# 06 — Evaluation

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Load the fine-tuned model (base + LoRA adapter)
2. Run inference on a sample of the held-out test set
3. Compute BLEU, ROUGE, and perplexity
4. Display Ground Truth vs. Prediction comparison table

Run this after `05_finetune_llm.ipynb`.


In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q evaluate rouge_score sacrebleu nltk pandas


In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import pandas as pd
from src.config import Config
from src.utils import load_jsonl, get_logger, detect_device
from src.evaluate_model import ModelEvaluator
from src.inference import EcommerceSupportBot

logger = get_logger("notebook06")
cfg = Config()
device = detect_device()


## 1. Load fine-tuned model (base + LoRA adapter)

In [ ]:
bot = EcommerceSupportBot(cfg)
model, tokenizer = bot.load_adapter_model()
print("Model loaded from adapter:", cfg.lora_adapter_dir)


## 2. Load test set and sample

In [ ]:
test_records = load_jsonl(cfg.processed_data_dir / "test.jsonl")
test_df = pd.DataFrame(test_records)

n_eval = min(cfg.eval_num_samples, len(test_df))
eval_df = test_df.sample(n_eval, random_state=cfg.seed).reset_index(drop=True)
print(f"Evaluating on {len(eval_df)} held-out test samples")
eval_df.head(3)


## 3. Generate predictions

In [ ]:
from tqdm.auto import tqdm

predictions = []
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    pred = bot.generate(
        row["instruction"],
        max_new_tokens=cfg.eval_max_new_tokens,
        temperature=0.3,   # lower temperature for more deterministic eval
        use_chat_template=False,  # match the raw instruction format used in training
    )
    predictions.append(pred)

eval_df["prediction"] = predictions
eval_df.head(3)


## 4. Compute BLEU, ROUGE, and Perplexity

In [ ]:
evaluator = ModelEvaluator()

report = evaluator.full_report(
    instructions=eval_df["instruction"].tolist(),
    ground_truths=eval_df["response"].tolist(),
    predictions=eval_df["prediction"].tolist(),
    model=model,
    tokenizer=tokenizer,
    full_texts_for_ppl=eval_df["text"].tolist() if "text" in eval_df.columns else None,
    device=device,
)

print("ROUGE:", report["rouge"])
print("BLEU:", report["bleu"]["score"])
print("Perplexity:", report["perplexity"])


## 5. Ground Truth vs. Prediction comparison table

In [ ]:
comparison_table = report["comparison_table"]
pd.set_option("display.max_colwidth", 200)
comparison_table.head(10)


In [ ]:
# Save the full comparison table + metrics for the README / portfolio writeup
out_dir = cfg.project_root / "data" / "processed"
comparison_table.to_csv(out_dir / "eval_comparison_table.csv", index=False)

metrics_summary = {
    "rouge1": report["rouge"].get("rouge1"),
    "rouge2": report["rouge"].get("rouge2"),
    "rougeL": report["rouge"].get("rougeL"),
    "bleu": report["bleu"].get("score"),
    "perplexity": report["perplexity"],
    "n_eval_samples": len(eval_df),
}
import json
with open(out_dir / "eval_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

print(metrics_summary)


## 6. Manual qualitative spot-check

Print a few full examples side-by-side for a closer read.

In [ ]:
for i in range(min(5, len(eval_df))):
    row = eval_df.iloc[i]
    print("=" * 80)
    print("INSTRUCTION:", row["instruction"])
    print("-" * 80)
    print("GROUND TRUTH:", row["response"])
    print("-" * 80)
    print("PREDICTION:  ", row["prediction"])
print("=" * 80)


## Summary

- Generated predictions for a sample of the held-out test set
- Computed BLEU, ROUGE-1/2/L, and perplexity
- Saved a full comparison table and metrics summary to `data/processed/`

Next: `07_inference.ipynb` to merge the adapter into a standalone model and build the final chat interface.